In [ ]:
import os 
import pandas as pd 
import numpy as np
import tensorflow as tf
import h5py
print(tf.__version__)  # Should show 2.13.0
import keras 
from keras import layers, models, Input
from keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:

df_data = pd.read_csv(filepath_or_buffer='PATH TO CSV FILE') 

In [ ]:
x_data = df_data.drop(columns=['key','quality'])
y_key_data = df_data['key']
y_quality_data = df_data['quality']


#split into train validate and test
x_train, x_test, y_key_train, y_key_test, y_quality_train, y_quality_test = train_test_split(x_data, 
    y_key_data, 
    y_quality_data, 
    test_size=0.2, 
    random_state=42)

print(x_test)
print(x_train)
x_test.to_numpy()
x_train.to_numpy()
y_key_test.to_numpy()
y_key_train.to_numpy()
y_quality_test.to_numpy()
y_quality_train.to_numpy()


#add padding 
padding = 0 


#make it into the correct shape

In [ ]:
def paddington(data,padVal, div=100):
    
    reminder = len(data)%div

    if reminder == 0:
        return data

    amount_to_pad = div-reminder

    if data.ndim == 1:
        padded = np.pad(data,(0,amount_to_pad),constant_values=padVal)
    else:
        padded = np.pad(data, ((0, amount_to_pad), (0, 0)), constant_values=padVal) 
    return padded       

In [ ]:
x_test_pad = paddington(x_test,padding)
x_train_pad = paddington(x_train,padding)
y_key_test = paddington(y_key_test,padding)
y_key_train = paddington(y_key_train,padding)
y_quality_test = paddington(y_quality_test,padding)
y_quality_train = paddington(y_quality_train,padding)

x_train_batches = x_train_pad.reshape(-1, 100, 25)
x_test_batches  = x_test_pad.reshape(-1, 100, 25)
y_key_train_batches = y_key_train.reshape(-1,100)
y_quality_train_batches = y_quality_train.reshape(-1,100)


In [ ]:
inputs = Input(shape=(100,25))
output = layers.Conv1D(filters=24, kernel_size=3,strides=1,padding="same",activation='relu')(inputs)
output = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(output)

keyInput= layers.Dense(32,activation='relu')(output)
key= layers.Dense(14,activation= 'softmax',name='key')(keyInput)

shapeInput = layers.Dense(32,activation='relu')(output)
quality = layers.Dense(4,activation='softmax',name='quality')(shapeInput)

model = models.Model(inputs=inputs, outputs = [key, quality])
model.compile(optimizer='adam',
              loss= 'sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
model.fit(x_train_batches, {'key': y_key_train_batches, 'quality': y_quality_train_batches}, 
                      batch_size=32, 
                      epochs=5)

In [ ]:
model.save(filepath='PATH TO MODEL SAVE DEST')

In [ ]:
df_to_pred = pd.read_csv(filepath_or_buffer='../data/McGill_Billboard/McGill-Billboard/0003/bothchromas.csv', header=None)

df_to_pred = df_to_pred.drop(columns=[0])
df_to_pred.to_numpy()
padded_pred = paddington(df_to_pred,padding)
padded_pred = padded_pred.reshape(-1,100,25)
predictions = model.predict(padded_pred)

key_probs = predictions[0].reshape(-1, predictions[0].shape[-1])
quality_probs = predictions[1].reshape(-1, predictions[1].shape[-1])

keys_final = np.argmax(key_probs,axis=-1)
quality_final = np.argmax(quality_probs,axis= -1)

print(keys_final)